# Modelo de Sugestões (multi-rótulo)

Este notebook treina APENAS o modelo de sugestões. Ele não depende do
modelo de perfil financeiro (não o retreina, não o carrega, não usa
`perfil_financeiro` como feature) — usa somente os dados brutos do usuário
(renda, valor investido, gastos por categoria) e os indicadores derivados
necessários para calcular os gatilhos de cada sugestão.

Um mesmo usuário pode receber 0, 1 ou várias sugestões simultaneamente,
por isso o rótulo (y) é um conjunto de colunas binárias (0/1), uma para
cada sugestão do catálogo — não uma única coluna categórica.


In [ ]:
from geradores import gera_dados_perfil_usuario

df_usuarios = gera_dados_perfil_usuario.run(4000)

In [ ]:
print('Trechos')
print(df_usuarios.head())

print('Shape')
print(df_usuarios.shape)

print('Potenciais falhas')
print(df_usuarios.isnull().sum())


Indicadores derivados (necessários para calcular os gatilhos das sugestões)


In [ ]:
colunas_gasto = df_usuarios.filter(like="gasto_").columns
df_usuarios['total_gasto'] = df_usuarios[colunas_gasto].sum(axis=1)

df_usuarios['percentual_gasto'] = df_usuarios['total_gasto'] / df_usuarios['renda_mensal']
df_usuarios['percentual_investido'] = df_usuarios['valor_investido'] / df_usuarios['renda_mensal']
df_usuarios['saldo'] = df_usuarios['renda_mensal'] - df_usuarios['total_gasto'] - df_usuarios['valor_investido']


Catálogo de sugestões e seus gatilhos (regras)


In [ ]:
LIMIARES_POR_CATEGORIA = {
    "gasto_alimentacao": ("monitorar_gasto_alimentacao", 0.20),
    "gasto_transporte": ("monitorar_gasto_transporte", 0.15),
    "gasto_saude": ("monitorar_gasto_saude", 0.15),
    "gasto_moradia": ("monitorar_gasto_moradia", 0.35),
    "gasto_educacao": ("monitorar_gasto_educacao", 0.15),
    "gasto_lazer": ("monitorar_gasto_lazer", 0.10),
    "gasto_servicos": ("monitorar_gasto_servicos", 0.7),
    "gasto_assinaturas": ("reduzir_assinaturas", 0.03),
    "gasto_dividas": ("monitorar_gasto_dividas", 0.20),
    "gasto_outras": ("monitorar_gasto_outras", 0.06),
}

COLUNAS_SUGESTOES = [nome for nome, _ in LIMIARES_POR_CATEGORIA.values()] + [
    "aumentar_investimento",
    "revisar_orcamento_urgente",
    "manter_bom_controle_financeiro",
]


def gerar_sugestoes(linha):
    sugestoes = {}

    # Sugestões por categoria: gasto_X / renda_mensal > limiar
    for coluna_gasto, (nome_sugestao, limiar) in LIMIARES_POR_CATEGORIA.items():
        percentual_categoria = linha[coluna_gasto] / linha['renda_mensal']
        sugestoes[nome_sugestao] = int(percentual_categoria > limiar)

    # Sugestões gerais
    sugestoes["aumentar_investimento"] = int(linha['percentual_investido'] < 0.05)
    sugestoes["revisar_orcamento_urgente"] = int(linha['saldo'] < 0)

    # Reforço positivo: só ativa se NENHUMA outra sugestão foi ativada
    nenhuma_ativada = sum(sugestoes.values()) == 0
    sugestoes["manter_bom_controle_financeiro"] = int(nenhuma_ativada)

    return pd.Series(sugestoes)


import pandas as pd

df_usuarios[COLUNAS_SUGESTOES] = df_usuarios.apply(gerar_sugestoes, axis=1)


Validação rápida: distribuição de cada sugestão (quantos usuários ativaram cada uma).
Fique de olho em sugestões muito raras (poucos exemplos positivos) — isso
pode prejudicar o aprendizado daquela sugestão específica.


In [ ]:
print(df_usuarios[COLUNAS_SUGESTOES].sum().sort_values(ascending=False))


Definir valores (x = dados brutos + indicadores derivados; y = as 13 colunas de sugestão)


In [ ]:
x = df_usuarios.drop(columns=COLUNAS_SUGESTOES)
y = df_usuarios[COLUNAS_SUGESTOES]


Dividir treino e teste

Observação: não usamos `stratify` aqui, pois o `train_test_split` não
oferece uma forma direta de estratificar múltiplos rótulos binários
simultaneamente. Com 2.000 usuários, a divisão aleatória tende a manter
proporções razoáveis mesmo assim.


In [ ]:
from sklearn.model_selection import train_test_split

x_treino, x_teste, y_treino, y_teste = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42
)

print("Tamanho treino:", x_treino.shape)
print("Tamanho teste:", x_teste.shape)


Treinar modelo


In [ ]:
from sklearn.ensemble import RandomForestClassifier

modelo_sugestoes = RandomForestClassifier(random_state=42)
modelo_sugestoes.fit(x_treino, y_treino)


Métricas (uma sugestão de cada vez, já que é multi-rótulo)


In [ ]:
from sklearn.metrics import classification_report

y_previsto = pd.DataFrame(
    modelo_sugestoes.predict(x_teste),
    columns=COLUNAS_SUGESTOES,
    index=x_teste.index,
)

for coluna in COLUNAS_SUGESTOES:
    print(f"--- {coluna} ---")
    print(classification_report(y_teste[coluna], y_previsto[coluna], zero_division=0))


Métrica resumida (Hamming Loss: em média, quantas sugestões o modelo erra por usuário)


In [ ]:
from sklearn.metrics import hamming_loss

print("Hamming Loss:", hamming_loss(y_teste, y_previsto))


Testes com casos manuais (usuários fabricados para conferir se as sugestões fazem sentido)


In [ ]:
casos_teste = pd.DataFrame([
    # Caso 1: gasto alto em Lazer e pouco investimento
    {
        "renda_mensal": 5000, "valor_investido": 100,
        "gasto_alimentação": 800, "gasto_transporte": 500, "gasto_saúde": 300,
        "gasto_moradia": 1200, "gasto_educação": 200, "gasto_lazer": 900,
        "gasto_serviços": 200, "gasto_assinaturas": 100, "gasto_dívidas": 300,
        "gasto_outras": 200,
    },
    # Caso 2: saldo negativo (orçamento estourado)
    {
        "renda_mensal": 4000, "valor_investido": 0,
        "gasto_alimentação": 900, "gasto_transporte": 600, "gasto_saúde": 400,
        "gasto_moradia": 1500, "gasto_educação": 300, "gasto_lazer": 400,
        "gasto_serviços": 300, "gasto_assinaturas": 150, "gasto_dívidas": 800,
        "gasto_outras": 300,
    },
    # Caso 3: perfil saudável, sem sinais de alerta (deve ativar "manter_bom_controle_financeiro")
    {
        "renda_mensal": 12000, "valor_investido": 3000,
        "gasto_alimentação": 700, "gasto_transporte": 400, "gasto_saúde": 400,
        "gasto_moradia": 1800, "gasto_educação": 300, "gasto_lazer": 300,
        "gasto_serviços": 200, "gasto_assinaturas": 80, "gasto_dívidas": 200,
        "gasto_outras": 150,
    },
])

colunas_gasto_teste = casos_teste.filter(like="gasto_").columns
casos_teste['total_gasto'] = casos_teste[colunas_gasto_teste].sum(axis=1)
casos_teste['percentual_gasto'] = casos_teste['total_gasto'] / casos_teste['renda_mensal']
casos_teste['percentual_investido'] = casos_teste['valor_investido'] / casos_teste['renda_mensal']
casos_teste['saldo'] = casos_teste['renda_mensal'] - casos_teste['total_gasto'] - casos_teste['valor_investido']

casos_teste_x = casos_teste[x.columns]
previsao_casos = pd.DataFrame(
    modelo_sugestoes.predict(casos_teste_x),
    columns=COLUNAS_SUGESTOES,
)

# Traduz o vetor de 0/1 de cada linha em uma lista de nomes de sugestões ativas
previsao_casos['sugestoes_ativas'] = previsao_casos.apply(
    lambda linha: [col for col in COLUNAS_SUGESTOES if linha[col] == 1], axis=1
)

print(previsao_casos[['sugestoes_ativas']])


Salvar modelo de sugestões


In [ ]:
import joblib

joblib.dump(modelo_sugestoes, "../servico-dados/modelos/modelo_sugestoes.joblib")
joblib.dump(list(x.columns), "../servico-dados/modelos/colunas_sugestoes.joblib")
joblib.dump(COLUNAS_SUGESTOES, "../servico-dados/modelos/nomes_sugestoes.joblib")
